# 06. 質問→採点→推薦パイプラインの自己検証

`docs/questions.md`で確定した10問を、実際に回答→採点→PCA→RIASEC逆変換→類似職業という
パイプラインに通して検証する。**採点ロジック（二択の回答を主成分スコアに変換する部分）は
符号の反転や標準化の扱いを間違えやすいので、質問文よりも先にここを疑う。** 人間が回答する
前に、まず符号が合っているかを合成テストケースで確認する。


In [1]:
import re

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

from ipd_loader import load_numeric

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

num, num_labels = load_numeric()
names = num[num.columns[1]]


## 167職業のRIASEC空間を再構成する（02/05と同じ手順）

In [2]:
riasec_cols = [c for c in num.columns if re.match(r"IPD_04_01_", str(c))]
know_cols = [c for c in num.columns if re.match(r"IPD_04_04_01_", str(c))]
work_cols = [c for c in num.columns if re.match(r"IPD_04_05_", str(c))]

riasec = num[riasec_cols].apply(pd.to_numeric, errors="coerce")
know = num[know_cols].apply(pd.to_numeric, errors="coerce")
work = num[work_cols].apply(pd.to_numeric, errors="coerce")

riasec_observed = ~riasec.isna().all(axis=1)
no_input = know.isna().all(axis=1) & work.isna().all(axis=1)
predictable = (~riasec_observed) & ~no_input


def scale_domain(domain_df):
    scaler = StandardScaler()
    filled = domain_df.fillna(domain_df.mean())
    scaled = pd.DataFrame(scaler.fit_transform(filled), columns=domain_df.columns, index=domain_df.index)
    return scaled.fillna(0.0)


know_scaled = scale_domain(know)
work_scaled = scale_domain(work)
X_all = pd.concat([know_scaled, work_scaled], axis=1)

X_train = X_all[riasec_observed]
y_train = riasec[riasec_observed]
riasec_model = Ridge(alpha=1.0).fit(X_train.values, y_train.values)

riasec_full = riasec.copy()
X_pred = X_all[predictable]
riasec_full.loc[predictable, riasec_cols] = riasec_model.predict(X_pred.values)

awareness = pd.read_csv("../data/processed/awareness_scores.csv", index_col=0)
name_to_numidx = pd.Series(num.index, index=names)

usable_numidx = []
for _, row in awareness[awareness["recommendable"]].iterrows():
    hit = name_to_numidx.get(row["職業名"])
    if hit is None:
        continue
    if isinstance(hit, pd.Series):
        hit = hit.iloc[0]
    usable_numidx.append(hit)

riasec_167 = riasec_full.loc[usable_numidx]
job_names_167 = names.loc[usable_numidx]
riasec_167.index = job_names_167.values

riasec_scaler = StandardScaler().fit(riasec.loc[riasec_observed, riasec_cols])
riasec_167_z = pd.DataFrame(
    riasec_scaler.transform(riasec_167.values), columns=riasec_cols, index=riasec_167.index
)

pca_full = PCA(n_components=6, random_state=0).fit(riasec_167_z.values)
print("167件の再構成:", riasec_167.shape)
print("寄与率:", pca_full.explained_variance_ratio_.round(3))


167件の再構成: (167, 6)
寄与率: [0.368 0.237 0.183 0.111 0.054 0.046]


/Users/oobasouma/yumetane/api/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 質問定義（`docs/questions.md`の対応表をそのままコードに落とす）

`positive_choice`が「前者」なら、その質問で前者を選んだときに+1、後者を選んだときに-1。
「後者」ならその逆。


In [3]:
QUESTIONS = [
    {"id": 1, "axis": "PC1", "positive_choice": "後者", "text": "テストのように答えが一つに決まっている問題(前者)と、感想文のように人それぞれ答えが違う問題(後者)、どっちが面白い？"},
    {"id": 2, "axis": "PC1", "positive_choice": "後者", "text": "教わったとおりにやる(前者)のと、自分なりのやり方を試す(後者)の、どっちが好き？"},
    {"id": 3, "axis": "PC1", "positive_choice": "後者", "text": "レシピどおりに正確に作る(前者)のと、レシピを見ながらも自分なりにアレンジする(後者)の、どっちが好き？"},
    {"id": 4, "axis": "PC2", "positive_choice": "前者", "text": "こわれたものを自分で直してみる(前者)のと、友達のけんかの仲直りを手伝う(後者)の、どっちをやってみたい？"},
    {"id": 5, "axis": "PC2", "positive_choice": "前者", "text": "新しい道具や機械のしくみを調べる(前者)のと、友達や家族の悩みを聞いてあげる(後者)の、どっちが得意な気がする？"},
    {"id": 6, "axis": "PC2", "positive_choice": "前者", "text": "作り方の動画を見て何かを作る(前者)のと、人の話を聞いてアドバイスする(後者)の、どっちが楽しそう？"},
    {"id": 7, "axis": "PC3", "positive_choice": "前者", "text": "最初に完成までの手順を決めてから進める(前者)のと、進めながら決めていく(後者)の、どっちが自分に近い？"},
    {"id": 8, "axis": "PC3", "positive_choice": "前者", "text": "ゲームは、ルールどおりに進める(前者)のと、自分たちでルールを変えて遊ぶ(後者)の、どっちが好き？"},
    {"id": 9, "axis": "PC4", "positive_choice": "前者", "text": "美術や音楽のように表現の美しさ・かっこよさにこだわる授業(前者)と、理科や数学のようにしくみや理由を突き止める授業(後者)、どっちが好き？"},
    {"id": 10, "axis": "PC4", "positive_choice": "前者", "text": "自由研究をやるなら、作品を作って見せる(前者)のと、調べて分かったことをまとめる(後者)の、どっちにする？"},
]

AXIS_N = {"PC1": 3, "PC2": 3, "PC3": 2, "PC4": 2}
assert sum(AXIS_N.values()) == 10
for axis, n in AXIS_N.items():
    assert sum(1 for q in QUESTIONS if q["axis"] == axis) == n
print("質問定義OK：10問、軸ごとの件数もdocs/questions.mdと一致")


質問定義OK：10問、軸ごとの件数もdocs/questions.mdと一致


## 採点ロジック

1. 軸ごとの生スコア = Σ(+1 if 正の選択 else -1)  → 範囲は軸の質問数に応じて[-n, +n]
2. 生スコアを、167職業の実際のPCAスコアのスケール（標準偏差）に線形マッピングする。
   「全問その方向に回答」を、その軸の職業分布のSCALE_SD倍（デフォルト1.5）に対応させる。
   これは仮の較正であり、実際の回答データが集まったら見直す前提（`docs/questions.md`に
   未解決課題として記載済み）
3. 4軸のスコアを6次元PCA空間の先頭4次元に置き、PC5・PC6は0（分布の平均）とする
4. `pca_full.inverse_transform`でRIASECの標準化空間に戻す


In [4]:
SCALE_SD = 1.5
pc_std = riasec_167_z.values @ pca_full.components_.T  # = pca_full.transform と同じ
pc_std = pc_std.std(axis=0)  # 各PCの標準偏差（166件基準）


def score_answers(answers: dict) -> dict:
    """answers: {question_id: "前者" or "後者"}"""
    raw = {axis: 0 for axis in AXIS_N}
    for q in QUESTIONS:
        choice = answers[q["id"]]
        sign = 1 if choice == q["positive_choice"] else -1
        raw[q["axis"]] += sign
    return raw


def raw_to_pc_vector(raw: dict) -> np.ndarray:
    pc_index = {"PC1": 0, "PC2": 1, "PC3": 2, "PC4": 3}
    vec = np.zeros(6)
    for axis, score in raw.items():
        n = AXIS_N[axis]
        frac = score / n  # -1..+1
        vec[pc_index[axis]] = frac * SCALE_SD * pc_std[pc_index[axis]]
    return vec


def answers_to_riasec(answers: dict):
    raw = score_answers(answers)
    pc_vec = raw_to_pc_vector(raw)
    riasec_z = pca_full.inverse_transform(pc_vec.reshape(1, -1))[0]
    return raw, pc_vec, riasec_z


def top_similar_jobs(riasec_z_vec, n=10):
    sims = cosine_similarity(riasec_z_vec.reshape(1, -1), riasec_167_z.values)[0]
    ranked = pd.Series(sims, index=riasec_167_z.index).sort_values(ascending=False)
    return ranked.head(n)


## 符号チェック：合成テストケース

各軸だけを極端な方向に回答した架空の回答者を作り、`top_similar_jobs`が
「各軸の高い方の職業リスト」（`05_question_design.ipynb`で確認済み）と一致する方向に
出るかを確認する。ズレていれば符号が反転しているというサイン。


In [5]:
def neutral_answers():
    """各軸を0（両端の中間）にする回答。二択なので厳密な0は作れないため、
    軸内の質問を半分ずつ「正」「負」に振って近似する。"""
    ans = {}
    for axis, n in AXIS_N.items():
        qs = [q for q in QUESTIONS if q["axis"] == axis]
        for i, q in enumerate(qs):
            other = "前者" if q["positive_choice"] == "後者" else "後者"
            ans[q["id"]] = q["positive_choice"] if i % 2 == 0 else other
    return ans


def set_axis(base: dict, axis: str, to_positive: bool) -> dict:
    ans = dict(base)
    for q in QUESTIONS:
        if q["axis"] == axis:
            other = "前者" if q["positive_choice"] == "後者" else "後者"
            ans[q["id"]] = q["positive_choice"] if to_positive else other
    return ans


base = neutral_answers()
print("中立回答のPCベクトル:", answers_to_riasec(base)[1].round(2))
print()

for axis in ["PC1", "PC2", "PC3", "PC4"]:
    for direction, to_positive in [("高い方(正)", True), ("低い方(負)", False)]:
        ans = set_axis(base, axis, to_positive)
        _, pc_vec, riasec_z = answers_to_riasec(ans)
        top3 = top_similar_jobs(riasec_z, n=3)
        print(f"{axis} {direction}: PCベクトル={pc_vec.round(2)}")
        print(f"  近い職業トップ3: {list(top3.index)}")
    print()


中立回答のPCベクトル: [0.73 0.58 0.   0.   0.   0.  ]

PC1 高い方(正): PCベクトル=[2.18 0.58 0.   0.   0.   0.  ]
  近い職業トップ3: ['学芸員', 'アウトドアインストラクター', '図書編集者']
PC1 低い方(負): PCベクトル=[-2.18  0.58  0.    0.    0.    0.  ]
  近い職業トップ3: ['野菜つけ物製造', '化学製品製造オペレーター', 'キッティング作業員（PCセットアップ作業員）']

PC2 高い方(正): PCベクトル=[0.73 1.75 0.   0.   0.   0.  ]
  近い職業トップ3: ['航空機開発エンジニア（ジェットエンジン）', '自動車技術者', 'ビール製造']
PC2 低い方(負): PCベクトル=[ 0.73 -1.75  0.    0.    0.    0.  ]
  近い職業トップ3: ['医薬情報担当者（MR）', '介護支援専門員/ケアマネジャー', 'キャリアカウンセラー/キャリアコンサルタント']

PC3 高い方(正): PCベクトル=[0.73 0.58 1.54 0.   0.   0.  ]
  近い職業トップ3: ['外科医', '職業訓練指導員', '獣医師']
PC3 低い方(負): PCベクトル=[ 0.73  0.58 -1.54  0.    0.    0.  ]
  近い職業トップ3: ['広報コンサルタント', '陶磁器技術者', '広告デザイナー']

PC4 高い方(正): PCベクトル=[0.73 0.58 0.   1.2  0.   0.  ]
  近い職業トップ3: ['フラワーデザイナー', 'ネイリスト', '造園工']
PC4 低い方(負): PCベクトル=[ 0.73  0.58  0.   -1.2   0.    0.  ]
  近い職業トップ3: ['ファインセラミックス製造技術者', 'バイオテクノロジー技術者', 'ファンドマネージャー']

